In [ ]:
# ==========================================================
# TFG
# Formulación y resolución de un modelo de optimización
# multiobjetivo para la planificación eficiente y sostenible
# de redes de transporte internacional de pasajeros bajo
# restricciones operativas.
#
# Universidad Internacional de Valencia (VIU)
#
# CAPÍTULO 4
# 4.5 Implementación computacional del modelo
# ==========================================================

In [ ]:
# ==========================================================
# LIBRERÍAS
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

print("=" * 60)
print("LIBRERÍAS CARGADAS CORRECTAMENTE")
print("=" * 60)

In [ ]:
# ==========================================================
# CARGA DE LA RED AÉREA
# ==========================================================

df = pd.read_excel("../datos/red_aerea.xlsx")

print("Número de arcos:", len(df))

df.head()

In [ ]:
# ==========================================================
# COMPROBACIÓN DE LA BASE DE DATOS
# ==========================================================

print("Dimensiones de la base de datos:", df.shape)

print("\nNombres de las columnas:")
print(df.columns.tolist())

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nTipos de datos:")
print(df.dtypes)

print("\nArcos duplicados:")
print(df["Arco"].duplicated().sum())

In [ ]:
# ==========================================================
# 4.5.1 CONJUNTOS E ÍNDICES
# ==========================================================

# Conjunto de nodos de la red
N = sorted(set(df["Origen"]).union(set(df["Destino"])))

# Conjunto de arcos físicos no dirigidos:
# A01, A02, ..., A28
A = df["Arco"].tolist()

# Diccionarios que relacionan cada arco físico
# con sus nodos de origen y destino
origen_arco = dict(zip(df["Arco"], df["Origen"]))
destino_arco = dict(zip(df["Arco"], df["Destino"]))

# Conjunto de arcos dirigidos (AD)
# Cada arco físico se representa mediante dos arcos dirigidos,
# uno en cada sentido.
AD = []

for a in A:
    i = origen_arco[a]
    j = destino_arco[a]

    AD.append((i, j))
    AD.append((j, i))

print("Número de nodos:", len(N))
print("Número de arcos físicos:", len(A))
print("Número de arcos dirigidos:", len(AD))

print("\nConjunto de nodos N:")
for nodo in N:
    print(nodo)

print("\nConjunto de arcos físicos A:")
print(A)

print("\nPrimeros arcos dirigidos AD:")
print(AD[:6])

In [ ]:
# ==========================================================
# 4.5.2 PARÁMETROS DEL MODELO
# ==========================================================

# ----------------------------------------------------------
# Parámetros asociados a los arcos físicos
# ----------------------------------------------------------

coste = dict(zip(df["Arco"], df["Coste medio (€)"]))
tiempo = dict(zip(df["Arco"], df["Tiempo medio (h)"]))
emisiones = dict(
    zip(df["Arco"], df["Emisiones medias (kg CO₂)"])
)
distancia = dict(
    zip(df["Arco"], df["Distancia media (km)"])
)
vuelos_observados = dict(
    zip(df["Arco"], df["N.º de vuelos observados"])
)

# ----------------------------------------------------------
# Parámetros asociados a los arcos dirigidos
# ----------------------------------------------------------

coste_d = {}
tiempo_d = {}
emisiones_d = {}
distancia_d = {}
arco_fisico = {}

for a in A:
    i = origen_arco[a]
    j = destino_arco[a]

    # Sentido i → j
    coste_d[i, j] = coste[a]
    tiempo_d[i, j] = tiempo[a]
    emisiones_d[i, j] = emisiones[a]
    distancia_d[i, j] = distancia[a]
    arco_fisico[i, j] = a

    # Sentido j → i
    coste_d[j, i] = coste[a]
    tiempo_d[j, i] = tiempo[a]
    emisiones_d[j, i] = emisiones[a]
    distancia_d[j, i] = distancia[a]
    arco_fisico[j, i] = a

print("Parámetros físicos creados:", len(coste))
print("Parámetros dirigidos creados:", len(coste_d))

print("=" * 60)
print("COMPROBACIÓN DE LOS PARÁMETROS (ARCO A01)")
print("=" * 60)
print("Origen:", origen_arco["A01"])
print("Destino:", destino_arco["A01"])
print("Coste medio (€):", coste["A01"])
print("Tiempo medio (h):", tiempo["A01"])
print(
    "Emisiones medias (kg CO₂):",
    emisiones["A01"]
)
print("Distancia media (km):", distancia["A01"])
print(
    "N.º de vuelos observados:",
    vuelos_observados["A01"]
)

In [ ]:
# ==========================================================
# COMPROBACIÓN DE LOS PARÁMETROS
# ==========================================================

# Comprobación de parámetros físicos
assert len(coste) == len(A)
assert len(tiempo) == len(A)
assert len(emisiones) == len(A)
assert len(distancia) == len(A)
assert len(vuelos_observados) == len(A)

# Comprobación de parámetros dirigidos
assert len(coste_d) == len(AD)
assert len(tiempo_d) == len(AD)
assert len(emisiones_d) == len(AD)
assert len(distancia_d) == len(AD)
assert len(arco_fisico) == len(AD)

print(
    "Comprobación superada: todos los arcos físicos y dirigidos "
    "tienen parámetros asociados."
)

In [ ]:
# ==========================================================
# 4.5.3 VARIABLES DE DECISIÓN
# ==========================================================

# Creación del modelo
modelo = gp.Model("Optimizacion_Red_Aerea_Multiobjetivo")

# Variable binaria dirigida:
# x[i,j] = 1 si el arco dirigido desde i hasta j
# forma parte del itinerario
# x[i,j] = 0 en caso contrario
x = modelo.addVars(
    AD,
    vtype=GRB.BINARY,
    name="x"
)

modelo.update()

print("Modelo dirigido creado correctamente.")
print("Número de variables binarias:", modelo.NumVars)

In [ ]:
# ==========================================================
# COMPROBACIÓN DE LAS VARIABLES DE DECISIÓN
# ==========================================================

print("Primeras variables binarias creadas:")

for i, j in AD[:5]:
    print(f"x[{i}, {j}]")

In [ ]:
# ==========================================================
# 4.5.4 FUNCIONES OBJETIVO
# ==========================================================

# Función objetivo 1:
# minimización del coste total
Z_coste = gp.quicksum(
    coste_d[i, j] * x[i, j]
    for i, j in AD
)

# Función objetivo 2:
# minimización del tiempo total
Z_tiempo = gp.quicksum(
    tiempo_d[i, j] * x[i, j]
    for i, j in AD
)

# Función objetivo 3:
# minimización de las emisiones totales
Z_emisiones = gp.quicksum(
    emisiones_d[i, j] * x[i, j]
    for i, j in AD
)

print(
    "Funciones objetivo construidas correctamente."
)

In [ ]:
# ==========================================================
# COMPROBACIÓN DE LAS FUNCIONES OBJETIVO
# ==========================================================

print("Número de términos de la función de coste:", Z_coste.size())
print("Número de términos de la función de tiempo:", Z_tiempo.size())
print("Número de términos de la función de emisiones:", Z_emisiones.size())

assert Z_coste.size() == len(AD)
assert Z_tiempo.size() == len(AD)
assert Z_emisiones.size() == len(AD)

print(
    "Validación superada: las tres funciones objetivo "
    "contienen los 56 arcos dirigidos."
)

In [ ]:
# ==========================================================
# DEFINICIÓN DEL ORIGEN Y DESTINO DEL ITINERARIO
# ==========================================================

nodo_origen = "Bangkok (BKK)"
nodo_destino = "Bali/Denpasar (DPS)"

print("Nodo de origen:", nodo_origen)
print("Nodo de destino:", nodo_destino)

In [ ]:
# ==========================================================
# PARÁMETRO DE BALANCE DE FLUJO
# ==========================================================

b = {}

for i in N:

    if i == nodo_origen:
        b[i] = 1

    elif i == nodo_destino:
        b[i] = -1

    else:
        b[i] = 0

print("Parámetro b creado correctamente.")
print("\nParámetro de balance de flujo:")

for nodo, valor in b.items():
    print(f"{nodo}: {valor}")

In [ ]:
# ==========================================================
# 4.5.5 RESTRICCIONES DEL MODELO
# ==========================================================

In [ ]:
# ==========================================================
# 4.5.5.1 CONSERVACIÓN DE FLUJO
# ==========================================================

for i in N:

    flujo_salida = gp.quicksum(
        x[u, v]
        for u, v in AD
        if u == i
    )

    flujo_entrada = gp.quicksum(
        x[u, v]
        for u, v in AD
        if v == i
    )

    modelo.addConstr(
        flujo_salida - flujo_entrada == b[i],
        name=f"flujo_{i}"
    )

modelo.update()

print("Restricciones de conservación de flujo creadas correctamente.")
print("Número de restricciones:", modelo.NumConstrs)

In [ ]:
# ==========================================================
# 4.5.5.2 RESTRICCIONES DE GRADO
# ==========================================================

for i in N:

    grado_salida = gp.quicksum(
        x[u, v]
        for u, v in AD
        if u == i
    )

    grado_entrada = gp.quicksum(
        x[u, v]
        for u, v in AD
        if v == i
    )

    modelo.addConstr(
        grado_salida <= 1,
        name=f"grado_salida_{i}"
    )

    modelo.addConstr(
        grado_entrada <= 1,
        name=f"grado_entrada_{i}"
    )

modelo.update()

print("Restricciones de grado creadas correctamente.")
print("Número total de restricciones:", modelo.NumConstrs)

In [ ]:
# ==========================================================
# 4.5.6 RESOLUCIÓN DE LOS PROBLEMAS MONOOBJETIVO
# ==========================================================

modelo.setObjective(Z_coste, GRB.MINIMIZE)

modelo.optimize()

In [ ]:
# ==========================================================
# RESULTADOS DE LA OPTIMIZACIÓN INDIVIDUAL DEL COSTE
# ==========================================================

if modelo.Status == GRB.OPTIMAL:

    # Recuperación de los arcos utilizados en la solución
    arcos_seleccionados = []

    for i, j in AD:
        if x[i, j].X > 0.5:
            arcos_seleccionados.append((i, j))

    # Presentación de los resultados
    print("=" * 60)
    print("RESULTADO DE LA OPTIMIZACIÓN INDIVIDUAL DEL COSTE")
    print("=" * 60)

    print("Estado: ÓPTIMO")
    print("Coste total (€):", round(modelo.ObjVal, 2))
    print("Número de arcos seleccionados:", len(arcos_seleccionados))

    print("\nArcos dirigidos seleccionados:")

    for i, j in arcos_seleccionados:
        print(f"{i} → {j}")

else:
    print("No se encontró una solución óptima.")
    print("Estado del modelo:", modelo.Status)

In [ ]:
# ==========================================================
# COMPROBACIÓN DE LA RUTA OBTENIDA
# ==========================================================

if modelo.Status == GRB.OPTIMAL:

    sucesores = {
        i: j
        for i, j in arcos_seleccionados
    }

    ruta = [nodo_origen]
    nodo_actual = nodo_origen

    while nodo_actual != nodo_destino:

        if nodo_actual not in sucesores:
            print("La solución no forma una ruta continua.")
            break

        nodo_actual = sucesores[nodo_actual]

        if nodo_actual in ruta:
            print("Se ha detectado un ciclo en la solución.")
            break

        ruta.append(nodo_actual)

    else:
        print("Ruta continua y sin ciclos:")
        print(" → ".join(ruta))

        if len(arcos_seleccionados) == len(ruta) - 1:
            print("No existen arcos desconectados adicionales.")
        else:
            print("Existen arcos seleccionados fuera de la ruta principal.")

In [ ]:
# ==========================================================
# INDICADORES DE LA RUTA DE COSTE MÍNIMO
# ==========================================================

coste_total = sum(
    coste_d[i, j]
    for i, j in arcos_seleccionados
)

tiempo_total = sum(
    tiempo_d[i, j]
    for i, j in arcos_seleccionados
)

emisiones_totales = sum(
    emisiones_d[i, j]
    for i, j in arcos_seleccionados
)

distancia_total = sum(
    distancia_d[i, j]
    for i, j in arcos_seleccionados
)

print("Coste total (€):", round(coste_total, 2))
print("Tiempo total (h):", round(tiempo_total, 2))
print("Emisiones totales (kg CO₂):", round(emisiones_totales, 2))
print("Distancia total (km):", round(distancia_total, 2))
print("Número de tramos:", len(arcos_seleccionados))
print("Número de conexiones intermedias:", max(0, len(arcos_seleccionados) - 1))

In [ ]:
# ==========================================================
# FUNCIÓN PARA EXTRAER Y VALIDAR UNA SOLUCIÓN
# ==========================================================

def obtener_resultados(nombre_objetivo):
    """
    Extrae la ruta y los indicadores de la solución óptima
    obtenida con el objetivo actualmente activo.
    """

    if modelo.Status != GRB.OPTIMAL:
        print(f"No se encontró solución óptima para {nombre_objetivo}.")
        return None

    # Arcos seleccionados por Gurobi
    arcos_solucion = [
        (i, j)
        for i, j in AD
        if x[i, j].X > 0.5
    ]

    # Construcción ordenada de la ruta
    sucesores = {
        i: j
        for i, j in arcos_solucion
    }

    ruta = [nodo_origen]
    nodo_actual = nodo_origen
    ciclo_detectado = False
    ruta_incompleta = False

    while nodo_actual != nodo_destino:

        if nodo_actual not in sucesores:
            ruta_incompleta = True
            break

        nodo_actual = sucesores[nodo_actual]

        if nodo_actual in ruta:
            ciclo_detectado = True
            break

        ruta.append(nodo_actual)

    # Indicadores totales
    coste_total = sum(
        coste_d[i, j]
        for i, j in arcos_solucion
    )

    tiempo_total = sum(
        tiempo_d[i, j]
        for i, j in arcos_solucion
    )

    emisiones_totales = sum(
        emisiones_d[i, j]
        for i, j in arcos_solucion
    )

    distancia_total = sum(
        distancia_d[i, j]
        for i, j in arcos_solucion
    )

    arcos_fuera_ruta = (
        len(arcos_solucion) != len(ruta) - 1
    )

    resultado = {
        "Objetivo": nombre_objetivo,
        "Ruta": " → ".join(ruta),
        "Coste (€)": round(coste_total, 2),
        "Tiempo (h)": round(tiempo_total, 2),
        "Emisiones (kg CO₂)": round(emisiones_totales, 2),
        "Distancia (km)": round(distancia_total, 2),
        "N.º de tramos": len(arcos_solucion),
        "N.º de conexiones": max(0, len(arcos_solucion) - 1),
        "Ciclo detectado": ciclo_detectado,
        "Ruta incompleta": ruta_incompleta,
        "Arcos fuera de la ruta": arcos_fuera_ruta
    }

    print(f"\nResultado: {nombre_objetivo}")
    print("Ruta:", resultado["Ruta"])
    print("Coste total (€):", resultado["Coste (€)"])
    print("Tiempo total (h):", resultado["Tiempo (h)"])
    print(
        "Emisiones totales (kg CO₂):",
        resultado["Emisiones (kg CO₂)"]
    )
    print("Distancia total (km):", resultado["Distancia (km)"])
    print("Número de tramos:", resultado["N.º de tramos"])
    print(
        "Número de conexiones:",
        resultado["N.º de conexiones"]
    )
    print("Ciclo detectado:", resultado["Ciclo detectado"])
    print("Ruta incompleta:", resultado["Ruta incompleta"])
    print(
        "Arcos fuera de la ruta:",
        resultado["Arcos fuera de la ruta"]
    )

    return resultado

In [ ]:
# ==========================================================
# SOLUCIÓN INDIVIDUAL 1: COSTE MÍNIMO
# ==========================================================

modelo.setObjective(Z_coste, GRB.MINIMIZE)
modelo.optimize()

resultado_coste = obtener_resultados(
    "Coste mínimo"
)

In [ ]:
# ==========================================================
# SOLUCIÓN INDIVIDUAL 2: TIEMPO MÍNIMO
# ==========================================================

modelo.setObjective(Z_tiempo, GRB.MINIMIZE)
modelo.optimize()

resultado_tiempo = obtener_resultados(
    "Tiempo mínimo"
)

In [ ]:
# ==========================================================
# SOLUCIÓN INDIVIDUAL 3: EMISIONES MÍNIMAS
# ==========================================================

modelo.setObjective(Z_emisiones, GRB.MINIMIZE)
modelo.optimize()

resultado_emisiones = obtener_resultados(
    "Emisiones mínimas"
)

In [ ]:
# ==========================================================
# COMPARACIÓN DE LAS SOLUCIONES INDIVIDUALES
# ==========================================================

resultados_individuales = pd.DataFrame([
    resultado_coste,
    resultado_tiempo,
    resultado_emisiones
])

columnas_comparacion = [
    "Objetivo",
    "Ruta",
    "Coste (€)",
    "Tiempo (h)",
    "Emisiones (kg CO₂)",
    "Distancia (km)",
    "N.º de tramos",
    "N.º de conexiones"
]

tabla_individuales = resultados_individuales[
    columnas_comparacion
]

print("=" * 70)
print("COMPARACIÓN DE LAS SOLUCIONES INDIVIDUALES")
print("=" * 70)

display(tabla_individuales)

In [ ]:
# ==========================================================
# 4.5.7 MATRIZ DE PAGOS Y VALORES DE REFERENCIA
# ==========================================================

matriz_pagos = pd.DataFrame(
    {
        "Solución optimizada": [
            "Coste mínimo",
            "Tiempo mínimo",
            "Emisiones mínimas"
        ],
        "Coste (€)": [
            resultado_coste["Coste (€)"],
            resultado_tiempo["Coste (€)"],
            resultado_emisiones["Coste (€)"]
        ],
        "Tiempo (h)": [
            resultado_coste["Tiempo (h)"],
            resultado_tiempo["Tiempo (h)"],
            resultado_emisiones["Tiempo (h)"]
        ],
        "Emisiones (kg CO₂)": [
            resultado_coste["Emisiones (kg CO₂)"],
            resultado_tiempo["Emisiones (kg CO₂)"],
            resultado_emisiones["Emisiones (kg CO₂)"]
        ]
    }
)

display(matriz_pagos)

In [ ]:
# ==========================================================
# VALORES IDEALES Y NADIR APROXIMADOS
# ==========================================================

ideal_coste = matriz_pagos["Coste (€)"].min()
ideal_tiempo = matriz_pagos["Tiempo (h)"].min()
ideal_emisiones = matriz_pagos["Emisiones (kg CO₂)"].min()

nadir_coste = matriz_pagos["Coste (€)"].max()
nadir_tiempo = matriz_pagos["Tiempo (h)"].max()
nadir_emisiones = matriz_pagos["Emisiones (kg CO₂)"].max()

print("=" * 60)
print("PUNTO IDEAL")
print("=" * 60)
print("Coste ideal (€):", ideal_coste)
print("Tiempo ideal (h):", ideal_tiempo)
print("Emisiones ideales (kg CO₂):", ideal_emisiones)

print("\n" + "=" * 60)
print("PUNTO NADIR APROXIMADO")
print("=" * 60)
print("Coste nadir (€):", nadir_coste)
print("Tiempo nadir (h):", nadir_tiempo)
print("Emisiones nadir (kg CO₂):", nadir_emisiones)

In [ ]:
# ==========================================================
# 4.5.8 APLICACIÓN DEL MÉTODO ε-CONSTRAINT
# ==========================================================

# Número de niveles utilizados para cada restricción
numero_niveles = 5

# Valores de ε distribuidos entre el ideal y el nadir aproximado
niveles_tiempo = np.linspace(
    ideal_tiempo,
    nadir_tiempo,
    numero_niveles
)

niveles_emisiones = np.linspace(
    ideal_emisiones,
    nadir_emisiones,
    numero_niveles
)

print("Niveles de ε para el tiempo:")
print(np.round(niveles_tiempo, 2))

print("\nNiveles de ε para las emisiones:")
print(np.round(niveles_emisiones, 2))

print(
    "\nNúmero inicial de combinaciones:",
    len(niveles_tiempo) * len(niveles_emisiones)
)

In [ ]:
# ============================================================
# EJECUCIÓN DEL MÉTODO ε-CONSTRAINT
# ============================================================

# Se minimiza el coste en todas las ejecuciones
modelo.setObjective(Z_coste, GRB.MINIMIZE)

# Evitar mostrar las 25 salidas completas de Gurobi
modelo.Params.OutputFlag = 0

# Reiniciar lista
soluciones_pareto = []

for epsilon_tiempo in niveles_tiempo:
    for epsilon_emisiones in niveles_emisiones:

        # Restricciones ε
        restriccion_tiempo = modelo.addConstr(
            Z_tiempo <= float(epsilon_tiempo),
            name="epsilon_tiempo"
        )

        restriccion_emisiones = modelo.addConstr(
            Z_emisiones <= float(epsilon_emisiones),
            name="epsilon_emisiones"
        )

        modelo.optimize()

        if modelo.Status == GRB.OPTIMAL:

            # Arcos seleccionados
            arcos_solucion = [
                (i, j)
                for i, j in AD
                if x[i, j].X > 0.5
            ]

            # Construcción de la ruta
            sucesores = {
                i: j
                for i, j in arcos_solucion
            }

            ruta = [nodo_origen]
            nodo_actual = nodo_origen

            ciclo_detectado = False
            ruta_incompleta = False

            while nodo_actual != nodo_destino:

                if nodo_actual not in sucesores:
                    ruta_incompleta = True
                    break

                nodo_actual = sucesores[nodo_actual]

                if nodo_actual in ruta:
                    ciclo_detectado = True
                    break

                ruta.append(nodo_actual)

            # Validación
            arcos_fuera_ruta = (
                len(arcos_solucion) != len(ruta) - 1
            )

            # Solo guardar soluciones válidas
            if (
                not ciclo_detectado
                and not ruta_incompleta
                and not arcos_fuera_ruta
            ):

                coste_total = sum(
                    coste_d[i, j]
                    for i, j in arcos_solucion
                )

                tiempo_total = sum(
                    tiempo_d[i, j]
                    for i, j in arcos_solucion
                )

                emisiones_totales = sum(
                    emisiones_d[i, j]
                    for i, j in arcos_solucion
                )

                distancia_total = sum(
                    distancia_d[i, j]
                    for i, j in arcos_solucion
                )

                soluciones_pareto.append({

                    "ε tiempo (h)": round(
                        float(epsilon_tiempo), 2
                    ),

                    "ε emisiones (kg CO₂)": round(
                        float(epsilon_emisiones), 2
                    ),

                    "Ruta": " → ".join(ruta),

                    "Coste (€)": round(
                        coste_total, 2
                    ),

                    "Tiempo (h)": round(
                        tiempo_total, 2
                    ),

                    "Emisiones (kg CO₂)": round(
                        emisiones_totales, 2
                    ),

                    "Distancia (km)": round(
                        distancia_total, 2
                    ),

                    "N.º de tramos": len(arcos_solucion),

                    "N.º de conexiones": max(
                        0,
                        len(arcos_solucion) - 1
                    ),

                    "Ciclo detectado": ciclo_detectado,
                    "Ruta incompleta": ruta_incompleta,
                    "Arcos fuera de la ruta": arcos_fuera_ruta

                })

        # Eliminar restricciones temporales
        modelo.remove(restriccion_tiempo)
        modelo.remove(restriccion_emisiones)
        modelo.update()

# Reactivar salida de Gurobi
modelo.Params.OutputFlag = 1

print("Combinaciones evaluadas:", 25)
print(
    "Soluciones factibles y válidas obtenidas:",
    len(soluciones_pareto)
)

In [ ]:
# ==========================================================
# TABLA DE SOLUCIONES ε-CONSTRAINT
# ==========================================================

pareto_df = pd.DataFrame(soluciones_pareto)

print("Número de soluciones almacenadas:")
print(len(pareto_df))

display(pareto_df)

In [ ]:
# ==========================================================
# 4.5.9 CONSTRUCCIÓN DE LA FRONTERA DE PARETO
# ==========================================================

pareto_unico = (
    pareto_df
    .drop_duplicates(
        subset=[
            "Ruta",
            "Coste (€)",
            "Tiempo (h)",
            "Emisiones (kg CO₂)"
        ]
    )
    .reset_index(drop=True)
)

print("Número de soluciones generadas:", len(pareto_df))
print("Número de soluciones únicas:", len(pareto_unico))

display(pareto_unico)

In [ ]:
# ==========================================================
# FILTRADO DE SOLUCIONES NO DOMINADAS
# ==========================================================

criterios = [
    "Coste (€)",
    "Tiempo (h)",
    "Emisiones (kg CO₂)"
]

indices_no_dominados = []

for i, solucion_i in pareto_unico.iterrows():

    dominada = False

    for j, solucion_j in pareto_unico.iterrows():

        if i == j:
            continue

        # La solución j es igual o mejor en todos los criterios
        mejor_o_igual = all(
            solucion_j[criterio] <= solucion_i[criterio]
            for criterio in criterios
        )

        # Y es estrictamente mejor en al menos un criterio
        estrictamente_mejor = any(
            solucion_j[criterio] < solucion_i[criterio]
            for criterio in criterios
        )

        if mejor_o_igual and estrictamente_mejor:
            dominada = True
            break

    if not dominada:
        indices_no_dominados.append(i)

pareto_final = (
    pareto_unico
    .loc[indices_no_dominados]
    .reset_index(drop=True)
)

print("Soluciones únicas:", len(pareto_unico))
print("Soluciones no dominadas:", len(pareto_final))

display(pareto_final)

In [ ]:
# ==========================================================
# IDENTIFICACIÓN DE LAS SOLUCIONES EFICIENTES
# ==========================================================

pareto_final = pareto_final.copy()

if "Solución" not in pareto_final.columns:
    pareto_final.insert(
        0,
        "Solución",
        [f"S{i+1}" for i in range(len(pareto_final))]
    )

display(
    pareto_final[
        [
            "Solución",
            "Ruta",
            "Coste (€)",
            "Tiempo (h)",
            "Emisiones (kg CO₂)"
        ]
    ]
)

In [ ]:
# ==========================================================
# COMPROBACIÓN DE LAS SOLUCIONES GENERADAS
# ==========================================================

print("Número total de soluciones eficientes:", len(pareto_final))

print("\nComprobación de restricciones:")

if pareto_final["Ciclo detectado"].sum() == 0:
    print("✓ Ninguna solución contiene ciclos.")

if pareto_final["Ruta incompleta"].sum() == 0:
    print("✓ Todas las rutas conectan correctamente origen y destino.")

if pareto_final["Arcos fuera de la ruta"].sum() == 0:
    print("✓ Ninguna solución contiene arcos desconectados.")

print("✓ Todas las soluciones almacenadas son factibles y válidas.")

In [ ]:
# ==========================================================
# RESUMEN DE LA EJECUCIÓN DEL ALGORITMO
# ==========================================================

print("=" * 60)
print("RESUMEN DE LA IMPLEMENTACIÓN")
print("=" * 60)

print(f"Número de nodos de la red: {len(N)}")
print(f"Número de arcos dirigidos: {len(AD)}")
print(f"Número de variables binarias: {modelo.NumVars}")
print(f"Número total de restricciones: {modelo.NumConstrs}")
print(
    f"Número de combinaciones ε evaluadas: "
    f"{len(niveles_tiempo) * len(niveles_emisiones)}"
)
print(
    f"Número de soluciones factibles y válidas: "
    f"{len(soluciones_pareto)}"
)
print(
    f"Número de soluciones no dominadas: "
    f"{len(pareto_final)}"
)

print("\nModelo ejecutado y validado correctamente.")

In [ ]:
# ==========================================================
# FIGURA 5.1
# FRONTERA DE PARETO: COSTE FRENTE A EMISIONES
# ==========================================================

fig1, ax = plt.subplots(figsize=(9, 6))

# Ordenar las soluciones por emisiones
pareto_ordenado = pareto_final.sort_values(
    "Emisiones (kg CO₂)"
)

# Línea de la frontera
ax.plot(
    pareto_ordenado["Emisiones (kg CO₂)"],
    pareto_ordenado["Coste (€)"],
    marker="o",
    markersize=8,
    linewidth=1.8
)

# Etiquetas de las soluciones
for _, fila in pareto_final.iterrows():
    ax.annotate(
        fila["Solución"],
        (
            fila["Emisiones (kg CO₂)"],
            fila["Coste (€)"]
        ),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold"
    )

# Título y ejes
ax.set_title(
    "Frontera de Pareto: coste frente a emisiones",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "Emisiones totales (kg CO₂ por pasajero)",
    fontsize=11
)

ax.set_ylabel(
    "Coste total del itinerario (€)",
    fontsize=11
)

# Cuadrícula
ax.grid(
    True,
    linestyle="--",
    linewidth=0.6,
    alpha=0.5
)

# Márgenes para evitar que las etiquetas queden cortadas
ax.margins(x=0.06, y=0.08)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# FIGURA 5.2
# FRONTERA DE PARETO: COSTE FRENTE A TIEMPO
# ==========================================================

fig2, ax = plt.subplots(figsize=(9, 6))

# Ordenar las soluciones por tiempo
pareto_ordenado = pareto_final.sort_values(
    "Tiempo (h)"
)

# Dibujar la frontera
ax.plot(
    pareto_ordenado["Tiempo (h)"],
    pareto_ordenado["Coste (€)"],
    marker="o",
    markersize=8,
    linewidth=1.8
)

# Etiquetas de las soluciones
for _, fila in pareto_final.iterrows():

    ax.annotate(
        fila["Solución"],
        (
            fila["Tiempo (h)"],
            fila["Coste (€)"]
        ),
        xytext=(8,8),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold"
    )

ax.set_title(
    "Frontera de Pareto: coste frente al tiempo",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel(
    "Tiempo total del itinerario (h)",
    fontsize=11
)

ax.set_ylabel(
    "Coste total del itinerario (€)",
    fontsize=11
)

ax.grid(
    True,
    linestyle="--",
    alpha=0.5
)

ax.margins(x=0.06, y=0.08)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# FIGURA 5.3
# FRONTERA DE PARETO: TIEMPO FRENTE A EMISIONES
# ==========================================================

fig3, ax = plt.subplots(figsize=(9, 6))

# Ordenar las soluciones por emisiones
pareto_ordenado = pareto_final.sort_values(
    "Emisiones (kg CO₂)"
)

# Dibujar la frontera
ax.plot(
    pareto_ordenado["Emisiones (kg CO₂)"],
    pareto_ordenado["Tiempo (h)"],
    marker="o",
    markersize=8,
    linewidth=1.8
)

# Etiquetas S1, S2, S3 y S4
for _, fila in pareto_final.iterrows():
    ax.annotate(
        fila["Solución"],
        (
            fila["Emisiones (kg CO₂)"],
            fila["Tiempo (h)"]
        ),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold"
    )

ax.set_title(
    "Frontera de Pareto: tiempo frente a emisiones",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "Emisiones totales (kg CO₂ por pasajero)",
    fontsize=11
)

ax.set_ylabel(
    "Tiempo total del itinerario (h)",
    fontsize=11
)

ax.grid(
    True,
    linestyle="--",
    linewidth=0.6,
    alpha=0.5
)

ax.margins(x=0.06, y=0.08)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# 4.5.10 NORMALIZACIÓN DE LOS CRITERIOS Y SELECCIÓN
# DE LA SOLUCIÓN DE COMPROMISO
# ==========================================================

criterios_pareto = [
    "Coste (€)",
    "Tiempo (h)",
    "Emisiones (kg CO₂)"
]

pareto_normalizado = pareto_final[
    ["Solución"] + criterios_pareto
].copy()

for criterio in criterios_pareto:

    valor_minimo = pareto_final[criterio].min()
    valor_maximo = pareto_final[criterio].max()

    pareto_normalizado[criterio] = (
        pareto_final[criterio] - valor_minimo
    ) / (
        valor_maximo - valor_minimo
    )

print(
    "Criterios normalizados correctamente "
    "en una escala entre 0 y 1."
)

display(pareto_normalizado)

In [ ]:
# ==========================================================
# FIGURA 5.4
# COMPARACIÓN NORMALIZADA DE LAS SOLUCIONES EFICIENTES
# ==========================================================

fig4, ax = plt.subplots(figsize=(10, 6))

pareto_normalizado.set_index(
    "Solución"
)[criterios_pareto].plot(
    kind="bar",
    ax=ax,
    width=0.75
)

ax.set_title(
    "Comparación normalizada de las soluciones eficientes",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "Solución eficiente",
    fontsize=11
)

ax.set_ylabel(
    "Valor normalizado",
    fontsize=11
)

ax.set_ylim(0, 1.10)

ax.legend(
    title="Criterio",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    ncol=3
)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.6,
    alpha=0.5
)

ax.tick_params(
    axis="x",
    rotation=0
)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# COMPROBACIÓN DE LAS FIGURAS
# ==========================================================

figuras_creadas = {
    "Figura 5.1": fig1,
    "Figura 5.2": fig2,
    "Figura 5.3": fig3,
    "Figura 5.4": fig4
}

print("Figuras disponibles para exportación:")

for nombre in figuras_creadas:
    print("✓", nombre)

In [ ]:
# ==========================================================
# DISTANCIA NORMALIZADA AL PUNTO IDEAL
# ==========================================================

import numpy as np

pareto_compromiso = pareto_normalizado.copy()

pareto_compromiso["Distancia al ideal"] = np.sqrt(
    pareto_compromiso["Coste (€)"] ** 2
    + pareto_compromiso["Tiempo (h)"] ** 2
    + pareto_compromiso["Emisiones (kg CO₂)"] ** 2
)

pareto_compromiso = pareto_compromiso.sort_values(
    "Distancia al ideal"
).reset_index(drop=True)

print("Clasificación de las soluciones eficientes según la distancia al punto ideal:")

display(pareto_compromiso)

In [ ]:
# ==========================================================
# SOLUCIÓN DE COMPROMISO IDENTIFICADA
# ==========================================================

id_compromiso = pareto_compromiso.loc[
    0,
    "Solución"
]

solucion_compromiso = pareto_final[
    pareto_final["Solución"] == id_compromiso
].copy()

print("Solución de compromiso obtenida:", id_compromiso)

display(
    solucion_compromiso[
        [
            "Solución",
            "Ruta",
            "Coste (€)",
            "Tiempo (h)",
            "Emisiones (kg CO₂)",
            "Distancia (km)",
            "N.º de tramos",
            "N.º de conexiones"
        ]
    ]
)

In [ ]:
# ============================================================
# 4.5.11 VALIDACIÓN DEL MODELO
# ============================================================

In [ ]:
# ============================================================
# VALIDACIÓN 1: COMPROBACIÓN DE CICLOS
# ============================================================

print("="*60)
print("VALIDACIÓN 1: COMPROBACIÓN DE CICLOS")
print("="*60)

if pareto_final["Ciclo detectado"].sum() == 0:
    print("✓ Ninguna solución contiene ciclos.")
else:
    print("✗ Se detectaron ciclos en alguna solución.")

In [ ]:
# ============================================================
# VALIDACIÓN 2: COMPROBACIÓN DE RUTAS INCOMPLETAS
# ============================================================

print("="*60)
print("VALIDACIÓN 2: COMPROBACIÓN DE RUTAS INCOMPLETAS")
print("="*60)

if pareto_final["Ruta incompleta"].sum() == 0:
    print("✓ Todas las soluciones llegan correctamente al destino.")
else:
    print("✗ Existen rutas incompletas.")

In [ ]:
# ============================================================
# VALIDACIÓN 3: COMPROBACIÓN DE EFICIENCIA DE PARETO
# ============================================================

print("=" * 60)
print("VALIDACIÓN 3: COMPROBACIÓN DE EFICIENCIA DE PARETO")
print("=" * 60)

criterios_validacion = [
    "Coste (€)",
    "Tiempo (h)",
    "Emisiones (kg CO₂)"
]

soluciones_dominadas = []

for i, solucion_i in pareto_final.iterrows():

    for j, solucion_j in pareto_final.iterrows():

        if i == j:
            continue

        mejor_o_igual = all(
            solucion_j[criterio] <= solucion_i[criterio]
            for criterio in criterios_validacion
        )

        estrictamente_mejor = any(
            solucion_j[criterio] < solucion_i[criterio]
            for criterio in criterios_validacion
        )

        if mejor_o_igual and estrictamente_mejor:
            soluciones_dominadas.append(
                solucion_i["Solución"]
            )
            break

if len(soluciones_dominadas) == 0:
    print(
        "✓ Todas las soluciones pertenecen a la "
        "frontera de Pareto."
    )
    print(
        "✓ Ninguna solución está dominada por otra."
    )
else:
    print(
        "✗ Se detectaron soluciones dominadas:",
        soluciones_dominadas
    )

In [ ]:
# ============================================================
# RESUMEN FINAL DE LA VALIDACIÓN
# ============================================================

validacion_final = pd.DataFrame({
    "Comprobación": [
        "Ausencia de ciclos",
        "Rutas completas",
        "Soluciones no dominadas",
        "Número de soluciones eficientes",
        "Solución de compromiso seleccionada"
    ],
    "Resultado": [
        "Validado" if pareto_final["Ciclo detectado"].sum() == 0 else "No validado",
        "Validado" if pareto_final["Ruta incompleta"].sum() == 0 else "No validado",
        "Validado" if len(soluciones_dominadas) == 0 else "No validado",
        len(pareto_final),
        id_compromiso
    ]
})

print("=" * 60)
print("VALIDACIÓN GLOBAL DEL MODELO")
print("=" * 60)

display(validacion_final)

print("\n✓ Validación global del modelo finalizada correctamente.")

In [ ]:
# ==========================================================
# 4.5.12 PREPARACIÓN DE LOS RESULTADOS PARA EXPORTACIÓN
# ==========================================================

# Tabla del punto ideal y nadir aproximado
tabla_ideal_nadir = pd.DataFrame({
    "Criterio": [
        "Coste (€)",
        "Tiempo (h)",
        "Emisiones (kg CO₂)"
    ],
    "Valor ideal": [
        ideal_coste,
        ideal_tiempo,
        ideal_emisiones
    ],
    "Valor nadir aproximado": [
        nadir_coste,
        nadir_tiempo,
        nadir_emisiones
    ]
})

# Tabla resumen de la implementación
resumen_implementacion = pd.DataFrame({
    "Indicador": [
        "Número de nodos",
        "Número de arcos físicos",
        "Número de arcos dirigidos",
        "Número de variables binarias",
        "Número de restricciones estructurales",
        "Combinaciones ε evaluadas",
        "Soluciones factibles",
        "Soluciones eficientes",
        "Solución de compromiso seleccionada"
    ],
    "Valor": [
        len(N),
        len(A),
        len(AD),
        modelo.NumVars,
        modelo.NumConstrs,
        len(niveles_tiempo) * len(niveles_emisiones),
        len(soluciones_pareto),
        len(pareto_final),
        id_compromiso
    ]
})

# Tabla de nodos
tabla_nodos = pd.DataFrame({
    "Nodo": N
})

# Tabla de arcos dirigidos y parámetros
tabla_arcos_dirigidos = pd.DataFrame([
    {
        "Arco físico": arco_fisico[i, j],
        "Origen": i,
        "Destino": j,
        "Coste medio (€)": coste_d[i, j],
        "Tiempo medio (h)": tiempo_d[i, j],
        "Emisiones medias (kg CO₂)": emisiones_d[i, j],
        "Distancia media (km)": distancia_d[i, j]
    }
    for i, j in AD
])

print("Tablas para exportación preparadas correctamente.")

In [ ]:
# ============================================================
# 4.5.13 EXPORTACIÓN AUTOMÁTICA DE TABLAS Y FIGURAS
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Rutas del proyecto
# ------------------------------------------------------------

carpeta_resultados = Path("../resultados")
carpeta_figuras = Path("../figuras")

# Crear las carpetas si no existen
carpeta_resultados.mkdir(parents=True, exist_ok=True)
carpeta_figuras.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Excel maestro con todas las hojas
# ------------------------------------------------------------

archivo_maestro = (
    carpeta_resultados / "resultados_modelo_aereo.xlsx"
)

with pd.ExcelWriter(
    archivo_maestro,
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        sheet_name="Base de datos",
        index=False
    )

    tabla_nodos.to_excel(
        writer,
        sheet_name="Nodos",
        index=False
    )

    tabla_arcos_dirigidos.to_excel(
        writer,
        sheet_name="Arcos dirigidos",
        index=False
    )

    tabla_individuales.to_excel(
        writer,
        sheet_name="Soluciones individuales",
        index=False
    )

    matriz_pagos.to_excel(
        writer,
        sheet_name="Matriz de pagos",
        index=False
    )

    tabla_ideal_nadir.to_excel(
        writer,
        sheet_name="Punto ideal y nadir",
        index=False
    )

    pareto_df.to_excel(
        writer,
        sheet_name="Resultados epsilon",
        index=False
    )

    pareto_final.to_excel(
        writer,
        sheet_name="Frontera de Pareto",
        index=False
    )

    pareto_normalizado.to_excel(
        writer,
        sheet_name="Valores normalizados",
        index=False
    )

    pareto_compromiso.to_excel(
        writer,
        sheet_name="Distancia al ideal",
        index=False
    )

    solucion_compromiso.to_excel(
        writer,
        sheet_name="Solución compromiso",
        index=False
    )

    validacion_final.to_excel(
        writer,
        sheet_name="Validación modelo",
        index=False
    )

    resumen_implementacion.to_excel(
        writer,
        sheet_name="Resumen implementación",
        index=False
    )


# ------------------------------------------------------------
# Archivos individuales para resultados y anexos
# ------------------------------------------------------------

pareto_df.to_excel(
    carpeta_resultados / "01_resultados_epsilon.xlsx",
    index=False
)

pareto_final.to_excel(
    carpeta_resultados / "02_frontera_pareto.xlsx",
    index=False
)

matriz_pagos.to_excel(
    carpeta_resultados / "03_matriz_pagos.xlsx",
    index=False
)

solucion_compromiso.to_excel(
    carpeta_resultados / "04_solucion_compromiso.xlsx",
    index=False
)

validacion_final.to_excel(
    carpeta_resultados / "05_validacion_modelo.xlsx",
    index=False
)


# ------------------------------------------------------------
# Exportación automática de las figuras en formato PNG
# ------------------------------------------------------------

fig1.savefig(
    carpeta_figuras / "Figura_5_1_Coste_vs_Emisiones.png",
    dpi=300,
    bbox_inches="tight"
)

fig2.savefig(
    carpeta_figuras / "Figura_5_2_Coste_vs_Tiempo.png",
    dpi=300,
    bbox_inches="tight"
)

fig3.savefig(
    carpeta_figuras / "Figura_5_3_Tiempo_vs_Emisiones.png",
    dpi=300,
    bbox_inches="tight"
)

fig4.savefig(
    carpeta_figuras / "Figura_5_4_Comparacion_Normalizada.png",
    dpi=300,
    bbox_inches="tight"
)


# ------------------------------------------------------------
# Resumen final de la exportación
# ------------------------------------------------------------

print("=" * 60)
print("EXPORTACIÓN FINALIZADA CORRECTAMENTE")
print("=" * 60)

print(f"Excel maestro: {archivo_maestro}")
print("Archivos individuales generados: 5")
print("Figuras exportadas: 4")
print(f"Carpeta de resultados: {carpeta_resultados}")
print(f"Carpeta de figuras: {carpeta_figuras}")